In [0]:
# ============================================================
#  NOTEBOOK 05 - RAW → SILVER: INVENTARIO
# ============================================================

STORAGE_ACCOUNT  = "stdatacolake"
CONTAINER_RAW    = "raw-zone"
CONTAINER_SILVER = "silver-zone"

RAW_PATH    = f"abfss://{CONTAINER_RAW}@{STORAGE_ACCOUNT}.dfs.core.windows.net/ORACLE"
SILVER_PATH = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/inventario"

In [0]:
# ── LEER CSV DESDE RAW ────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

schema_inventario = StructType([
    StructField("id_movimiento",      IntegerType(),  nullable=False),
    StructField("fecha_movimiento",   TimestampType(),nullable=True),
    StructField("id_producto",        IntegerType(),  nullable=True),
    StructField("id_bodega",          IntegerType(),  nullable=True),
    StructField("tipo_movimiento",    StringType(),   nullable=True),
    StructField("cantidad",           IntegerType(),  nullable=True),
    StructField("stock_resultante",   IntegerType(),  nullable=True),
    StructField("fecha_vencimiento",  TimestampType(),nullable=True),
])

df_raw = spark.read \
    .schema(schema_inventario) \
    .option("header", "true") \
    .csv(f"{RAW_PATH}/Datos_inventario.csv")

print(f"✅ Raw leído: {df_raw.count()} filas")
df_raw.show(5)

In [0]:
# ── LIMPIEZA Y TRANSFORMACIONES ───────────────────────────────
from pyspark.sql.functions import col, trim, when, datediff, to_date, current_date, current_timestamp

df_silver = df_raw \
    .filter(col("id_movimiento").isNotNull()) \
    .filter(col("id_producto").isNotNull()) \
    .filter(col("id_bodega").isNotNull()) \
    .filter(col("cantidad") > 0) \
    .withColumn("tipo_movimiento", trim(col("tipo_movimiento"))) \
    .withColumn("dias_para_vencer",
        datediff(to_date(col("fecha_vencimiento")), current_date())
    ) \
    .withColumn("alerta_vencimiento",
        when(col("dias_para_vencer") <= 30,  "CRITICO")
       .when(col("dias_para_vencer") <= 90,  "PROXIMO")
       .otherwise("OK")
    ) \
    .dropDuplicates(["id_movimiento"]) \
    .withColumn("_updated_at", current_timestamp())

print(f"✅ Silver listo: {df_silver.count()} filas")
df_silver.show(5)

In [0]:
# ── GUARDAR COMO DELTA EN SILVER ─────────────────────────────
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(SILVER_PATH)

print(f"✅ Guardado en Silver: {SILVER_PATH}")